In [1]:
# load the dataset
from pathlib import Path

from battery_aging.data_preprocessing import load_dataset

Dataset = "HUST"
data_folder = Path("..", "Battery_life_Dataset", Dataset)


datasets = load_dataset(data_folder, n_files=2, seed=42)

print([d["cell_id"] for d in datasets])

['HUST_10-7', 'HUST_1-4']


In [ ]:
# reduce the number of cycles in the dataset to speed up the analysis based 
# on the specified step size and the cycles to keep

from battery_aging.data_preprocessing import reduce_cycles

keep_cycles=[5, 10, 15, 30]
step = 20

for dataset in datasets:
    dataset["cycle_data"] = reduce_cycles(dataset["cycle_data"], step=step, keep_cycles=keep_cycles)


for dataset in datasets:
    print(
        dataset["cell_id"],
        len(dataset["cycle_data"]),
        [c["cycle_number"] for c in dataset["cycle_data"][:10]]
    )

HUST_10-7 97 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
HUST_1-4 83 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]


In [3]:
# First round of feature engineering

from battery_aging.feature_engineering import feature_engineering_1

features_df = feature_engineering_1(datasets)



In [4]:
for d in datasets:
    print(d["cell_id"], len(d["cycle_data"]))

print(len(datasets[0]["cycle_data"]))

HUST_10-7 97
HUST_1-4 83
97


In [5]:
# Second round of feature engineering

from battery_aging.feature_engineering import feature_engineering_2

features_df_2 = feature_engineering_2(datasets)
print(features_df_2.columns)
print(features_df_2.head())

features_df = features_df.merge(
    features_df_2,
    on=["cell_id", "cycle_number"],
    how="left"
)

Index(['cell_id', 'cycle_number', 'V_range', 'V_slope_mean', 'V_curvature',
       'V_n_peaks', 'RMSE_V', 'AreaDiff_V', 'Corr_V', 'MaxDev_V',
       'SlopeRMSE_V'],
      dtype='str')
     cell_id  cycle_number  V_range  V_slope_mean  V_curvature  V_n_peaks  \
0  HUST_10-7             1   1.6087      0.004049     0.002095          5   
1  HUST_10-7             2   1.6000      0.004600     0.002300          8   
2  HUST_10-7             3   1.6003      0.004597     0.002274         19   
3  HUST_10-7             4   1.6090      0.004592     0.002289         18   
4  HUST_10-7             5   1.5994      0.004547     0.002321         14   

     RMSE_V  AreaDiff_V    Corr_V  MaxDev_V  SlopeRMSE_V  
0  0.147322    0.063800  0.833745    1.0007     0.016584  
1  0.000000    0.000000  1.000000    0.0000     0.000000  
2  0.016257    0.003889  0.997839    0.2558     0.010565  
3  0.018768    0.006928  0.998411    0.1283     0.005973  
4  0.052997    0.016953  0.983999    0.3432     0.011979  

In [8]:
# third round of feature engineering: DTW features based on current curves

from battery_aging.feature_engineering import feature_engineering_dtw_current

dtw_df = feature_engineering_dtw_current(datasets)

print(dtw_df.columns)
print(dtw_df.head())

features_df = features_df.merge(
    dtw_df,
    on=["cell_id", "cycle_number"],
    how="left"
)

Index(['cell_id', 'cycle_number', 'DTW_I'], dtype='str')
     cell_id  cycle_number     DTW_I
0  HUST_10-7             1  0.710713
1  HUST_10-7             2  0.000000
2  HUST_10-7             3  0.053206
3  HUST_10-7             4  0.033351
4  HUST_10-7             5  0.108091


In [9]:
# fourth round of feature engineering: DTW features based on voltage curves

from battery_aging.feature_engineering import feature_engineering_dtw_voltage

dtw_v_df = feature_engineering_dtw_voltage(datasets)

print(dtw_v_df.columns)
print(dtw_v_df.head())

features_df = features_df.merge(
    dtw_v_df,
    on=["cell_id", "cycle_number"],
    how="left"
)

Index(['cell_id', 'cycle_number', 'DTW_V'], dtype='str')
     cell_id  cycle_number     DTW_V
0  HUST_10-7             1  1.637140
1  HUST_10-7             2  0.000000
2  HUST_10-7             3  0.315661
3  HUST_10-7             4  0.365062
4  HUST_10-7             5  0.352751


In [10]:
# Display the columns of the merged DataFrame and a sample of the new features

print(features_df.columns.tolist())
# print(features_df [
#     [
#         "V_range",
#         "V_slope_mean",
#         "V_curvature",
#         "V_n_peaks"
#     ]
# ].head())
print(features_df.head())

['cell_id', 'cycle_number', 'SOH', 'I_mean', 'I_std', 'charge_duration', 'discharge_duration', 'V_mean', 'V_std', 'V_range', 'V_slope_mean', 'V_curvature', 'V_n_peaks', 'RMSE_V', 'AreaDiff_V', 'Corr_V', 'MaxDev_V', 'SlopeRMSE_V', 'DTW_I', 'DTW_V']
     cell_id  cycle_number       SOH    I_mean     I_std  charge_duration  \
0  HUST_10-7             1  1.003048  0.022762  2.794810           2071.0   
1  HUST_10-7             2  1.000000 -0.008074  2.861839           1879.0   
2  HUST_10-7             3  1.001530 -0.011895  2.865007           1871.0   
3  HUST_10-7             4  1.003068 -0.009953  2.859391           1885.0   
4  HUST_10-7             5  1.004106 -0.014097  2.852216           1900.0   

   discharge_duration    V_mean     V_std  V_range  V_slope_mean  V_curvature  \
0              1775.0  3.312111  0.257952   1.6087      0.004049     0.002095   
1              1769.0  3.295809  0.248238   1.6000      0.004600     0.002300   
2              1772.0  3.297725  0.242523   1.

In [ ]:
import pickle
from pathlib import Path

# Zielordner
output_dir = Path("..", "engineered_data")
output_dir.mkdir(exist_ok=True)

# Dateiname
filename = output_dir / f"processed_battery_features{Dataset}.pkl"

# Speichern
with open(filename, "wb") as f:
    pickle.dump(features_df, f)

print(f"Saved to: {filename.resolve()}")

Saved to: C:\Users\Christoph\Documents\VS_Studio_Projekte\VS_Studio_Projekte\engineered_data\processed_battery_features_new_HUST.pkl
